# 🎬 VIDEO QUIZ GENERATOR STUDIO — GOOGLE COLAB GPU RUNNER
Notebook này khởi chạy toàn bộ Studio (Backend API + Remotion Engine + Web Editor) trực tiếp trên Google Colab có GPU miễn phí.

> ⚡ **LƯU Ý TỐI QUAN TRỌNG VỀ PHẦN CỨNG (GPU vs TPU):**
> - Hệ thống render video Remotion chạy trên **NVIDIA GPU (T4 GPU)** với bộ giải mã phần cứng **NVENC** (tốc độ **25 - 35+ FPS**).
> - **TUYỆT ĐỐI KHÔNG CHỌN TPU:** TPU (Tensor Processing Unit) là chip AI chỉ dành cho ma trận Deep Learning, **KHÔNG hỗ trợ đồ họa, Chromium WebGL hay FFmpeg** (sẽ báo lỗi *'Không thể kết nối với phần phụ trợ TPU'* và tụt xuống ~0.8 FPS).
> - Để chọn GPU đúng: Vào menu **Thời gian chạy (Runtime)** ➔ **Thay đổi loại thời gian chạy (Change runtime type)** ➔ Chọn **T4 GPU**.

In [ ]:
# ==============================================================================
# BƯỚC 0: KIỂM TRA PHẦN CỨNG & BỘ TĂNG TỐC (GPU / CPU / TPU DIAGNOSTIC)
# ==============================================================================
import os
import sys
import shutil
import subprocess

print("=" * 72)
print("🔍 ĐANG KIỂM TRA PHẦN CỨNG & BỘ TĂNG TỐC GOOGLE COLAB...")
print("=" * 72)

has_gpu = False
gpu_info = ""

if shutil.which("nvidia-smi"):
    try:
        raw_smi = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
            stderr=subprocess.STDOUT
        ).decode().strip()
        if raw_smi:
            gpu_name, gpu_mem, driver_ver = [x.strip() for x in raw_smi.split(",")]
            has_gpu = True
            print(f"\n✅ [XÁC NHẬN GPU]: {gpu_name} (VRAM: {gpu_mem})")
            print(f"✅ [NVIDIA DRIVER]: Phiên bản {driver_ver}")
            print(f"🚀 [TRẠNG THÁI]: Bộ tăng tốc phần cứng T4 GPU ĐÃ HOẠT ĐỘNG! Tốc độ dự kiến: 25 - 35+ FPS.\n")
    except Exception:
        pass

if not has_gpu:
    is_tpu = "COLAB_TPU_ADDR" in os.environ or "TPU_NAME" in os.environ
    if is_tpu:
        print("\n❌ [PHÁT HIỆN BẠN ĐANG CHỌN TPU - KHÔNG THỂ DÙNG ĐỂ RENDER VIDEO!]")
        print("   * TPU (Tensor Processing Unit) là chip toán học cho Deep Learning, KHÔNG CÓ GPU / WebGL / NVENC.")
        print("   * Đó là lý do bạn gặp lỗi 'Không thể kết nối với phần phụ trợ TPU' hoặc bị đẩy về CPU (~0.8 FPS).")
    else:
        print("\n⚠️ [CẢNH BÁO: BẠN ĐANG CHẠY TRÊN CPU (CHƯA BẬT GPU)!]")
        print("   * Google Colab CPU chỉ cấp 2 vCPUs yếu. Tốc độ render sẽ rất chậm (~0.8 - 2 FPS).")

    print("\n👉 HƯỚNG DẪN 3 BƯỚC ĐỂ BẬT GPU T4 (MIỄN PHÍ - ĐẠT 30+ FPS):")
    print("   1. Nhìn lên thanh menu phía trên: Bấm vào 'Thời gian chạy' (Runtime).")
    print("   2. Chọn 'Thay đổi loại thời gian chạy' (Change runtime type).")
    print("   3. Tại mục 'Bộ tăng tốc phần cứng' (Hardware accelerator), hãy chọn: 'T4 GPU'.")
    print("      (CHÚ Ý: Không chọn TPU vì TPU không hỗ trợ render video!)")
    print("   4. Bấm 'Lưu' (Save), sau đó chạy lại từ Bước 0 này.\n")
    print("=" * 72)


In [ ]:
# ==============================================================================
# BƯỚC 1: CLONE / CẬP NHẬT SOURCE CODE TỪ GITHUB
# ==============================================================================
import os
import shutil

# --- [CẤU HÌNH GITHUB]: Điền thông tin repository của bạn tại đây ---
GITHUB_REPO = "YOUR_USERNAME/YOUR_REPOSITORY"  # Ví dụ: "baomars/Video-quiz-new"
GITHUB_TOKEN = "YOUR_GITHUB_PERSONAL_ACCESS_TOKEN"  # Điền Personal Access Token nếu là Private Repo

# ------------------------------------------------------------------------------
REPO_NAME = GITHUB_REPO.split('/')[-1].replace('.git', '') if '/' in GITHUB_REPO and GITHUB_REPO != "YOUR_USERNAME/YOUR_REPOSITORY" else "Video-quiz-new"
WORKSPACE_DIR = f"/content/{REPO_NAME}"

print(f"[*] Thư mục làm việc: {WORKSPACE_DIR}")

if os.path.exists(WORKSPACE_DIR) and os.path.exists(os.path.join(WORKSPACE_DIR, 'package.json')):
    print(f"[✓] Thư mục dự án đã tồn tại. Tiến hành kéo code mới nhất (git pull)...")
    %cd {WORKSPACE_DIR}
    !git pull origin main || !git pull origin master || true
else:
    %cd /content
    if GITHUB_TOKEN and GITHUB_TOKEN != "YOUR_GITHUB_PERSONAL_ACCESS_TOKEN":
        AUTH_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    else:
        AUTH_URL = f"https://github.com/{GITHUB_REPO}.git"
    
    !git clone {AUTH_URL} {REPO_NAME} || echo "[!] Clone thất bại hoặc repo đã tồn tại."
    if os.path.exists(WORKSPACE_DIR):
        %cd {WORKSPACE_DIR}

print(f"\n[✓] Hiện đang tại: {os.getcwd()}")


In [ ]:
# ==============================================================================
# BƯỚC 2: CÀI ĐẶT DEPENDENCY & TĂNG TỐC HỆ THỐNG (CHROME, NVENC, XVFB, GPU EGL)
# ==============================================================================
import subprocess
import shutil
import os
import sys

print("=== BẮT ĐẦU CÀI ĐẶT & TỐI ƯU MÔI TRƯỜNG LINUX GPU ===")

# Thiết lập đường dẫn driver NVIDIA cho toàn bộ tiến trình con (FFmpeg, Chromium, Node.js)
nvidia_lib_path = "/usr/lib64-nvidia:/usr/local/cuda/lib64"
os.environ["LD_LIBRARY_PATH"] = f"{nvidia_lib_path}:{os.environ.get('LD_LIBRARY_PATH', '')}"

# 1. Cài đặt Node.js 20 LTS (nếu chưa có hoặc cũ)
node_ok = False
if shutil.which("node"):
    try:
        n_v = subprocess.check_output(["node", "-v"]).decode().strip()
        if int(n_v.lstrip('v').split('.')[0]) >= 18:
            print(f"[✓] Node.js sẵn sàng: {n_v}")
            node_ok = True
    except Exception:
        pass

if not node_ok:
    print("[*] Đang cài đặt Node.js v20 LTS...")
    !curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
    !apt-get install -y -qq nodejs > /dev/null 2>&1
    print("[✓] Node.js v20 đã được cài đặt thành công.")

# 2. Cài đặt FFmpeg, Xvfb & Drivers đồ họa tăng tốc phần cứng EGL/GLX
print("[*] Đang cài đặt FFmpeg, Xvfb và GPU Mesa EGL Drivers... Có thể mất 15-20s...")
!apt-get update -qq && !apt-get install -y -qq ffmpeg xvfb libegl1-mesa libgl1-mesa-glx libgl1-mesa-dri > /dev/null 2>&1
print("[✓] FFmpeg, Xvfb và GPU Drivers đã sẵn sàng.")

# 3. Cài đặt Python edge-tts cho giọng đọc tự nhiên
try:
    import edge_tts
    print("[✓] edge-tts đã có sẵn.")
except ImportError:
    print("[*] Đang cài đặt edge-tts...")
    !pip install -q edge-tts
    print("[✓] edge-tts đã cài đặt.")

# 4. Cài đặt Node modules
if os.path.exists("node_modules") and os.path.exists("node_modules/remotion"):
    print("[✓] node_modules đã tồn tại. Bỏ qua cài đặt lại để tiết kiệm thời gian.")
else:
    print("[*] Đang cài đặt các thư viện Node.js (khoảng 1-2 phút)... Cho phép caching...")
    !npm install --prefer-offline --no-audit --loglevel=error
    print("[✓] Cài đặt npm packages hoàn tất!")

# Cấp quyền thực thi cho Remotion Compositor FFmpeg trên Linux
!chmod +x node_modules/@remotion/compositor-linux-x64-gnu/ffmpeg 2>/dev/null || true

# 5. Tải trước Chrome Headless Shell cho Remotion
print("[*] Đang chuẩn bị Chrome Headless Shell siêu tốc cho Remotion...")
!node -e "import('@remotion/renderer').then(r => r.ensureBrowser({ chromeMode: 'headless-shell' })).then(res => console.log('[✓] Chrome Headless Shell sẵn sàng tại:', res.path)).catch(e => console.log('[!] Browser info:', e.message))"

# 6. Kiểm tra trực tiếp khả năng giải mã / mã hóa phần cứng NVENC
compositor_ffmpeg = "node_modules/@remotion/compositor-linux-x64-gnu/ffmpeg"
if os.path.exists(compositor_ffmpeg):
    !chmod +x {compositor_ffmpeg}
    test_nvenc = subprocess.run(
        [compositor_ffmpeg, "-f", "lavfi", "-i", "color=c=black:s=64x64:d=0.05", "-c:v", "h264_nvenc", "-f", "null", "-"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        env=os.environ.copy()
    )
    if test_nvenc.returncode == 0:
        print("\n🔥 [XÁC NHẬN THÀNH CÔNG]: CHIP MÃ HÓA PHẦN CỨNG NVIDIA NVENC HOẠT ĐỘNG HOÀN HẢO!")
        print("   Mọi tác vụ encode video sẽ chạy trực tiếp trên GPU, giải phóng 100% CPU.\n")
    else:
        print("\nℹ️ [THÔNG BÁO]: NVENC chưa được kích hoạt, hệ thống sẽ sử dụng CPU libx264 rất nhanh (veryfast).\n")

print("✅ TOÀN BỘ MÔI TRƯỜNG ĐÃ SẴN SÀNG ĐỂ RENDER TỐC ĐỘ CAO!")


In [ ]:
# ==============================================================================
# BƯỚC 3: KHỞI CHẠY STUDIO VỚI VIRTUAL DISPLAY XVFB & TẠO ĐƯỜNG HẦN PUBLIC
# ==============================================================================
import subprocess
import time
import urllib.request
import urllib.error
import re
import os
import shutil
from IPython.display import display, HTML

# 1. CẤU HÌNH CỔNG VÀ VIRTUAL DISPLAY CHO GPU RASTERIZATION
FRONTEND_PORT = int(os.environ.get("FRONTEND_PORT", 4500))
BACKEND_PORT = int(os.environ.get("BACKEND_PORT", 5410))
os.environ["FRONTEND_PORT"] = str(FRONTEND_PORT)
os.environ["BACKEND_PORT"] = str(BACKEND_PORT)

# Khởi động Xvfb Virtual Display để Chromium tận dụng GPU trên môi trường headless Linux
!pkill -f Xvfb > /dev/null 2>&1 || true
subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1280x720x24", "-ac"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
os.environ["DISPLAY"] = ":99"
nvidia_lib_path = "/usr/lib64-nvidia:/usr/local/cuda/lib64"
os.environ["LD_LIBRARY_PATH"] = f"{nvidia_lib_path}:{os.environ.get('LD_LIBRARY_PATH', '')}"
time.sleep(1)
print(f"[✓] Virtual Display Xvfb (:99) đã được bật cho Chromium GPU Rasterization!")

print("=" * 72)
print(f"[*] KHỞI CHẠY STUDIO VIDEO TRÊN CỔNG: {FRONTEND_PORT} (Backend: {BACKEND_PORT})")
print("=" * 72)

# Dọn dẹp process cũ trên các cổng
!fuser -k {FRONTEND_PORT}/tcp > /dev/null 2>&1 || true
!fuser -k 5400/tcp > /dev/null 2>&1 || true
!fuser -k {BACKEND_PORT}/tcp > /dev/null 2>&1 || true
!pkill -f cloudflared > /dev/null 2>&1 || true
!pkill -f localtunnel > /dev/null 2>&1 || true

# Khởi động Backend & Frontend ở chế độ background
server_log_path = "/content/studio_server.log"
server_log = open(server_log_path, "w")
server_proc = subprocess.Popen(
    ["npm", "run", "dev"],
    stdout=server_log,
    stderr=subprocess.STDOUT,
    shell=False,
    env=os.environ.copy()
)

# 2. CHỜ SERVER THỰC SỰ READY TRƯỚC KHI TẠO TUNNEL
def probe_port(port):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{port}", timeout=1.5) as resp:
            if resp.status in [200, 301, 302, 304]:
                return True
    except Exception:
        pass
    return False

print("[*] Đang chờ Web Server khởi động và sẵn sàng (READY)...")
active_port = None
for attempt in range(45):
    time.sleep(2)
    for p in [FRONTEND_PORT, 5400]:
        if probe_port(p):
            active_port = p
            break
    if active_port:
        break
    if (attempt + 1) % 5 == 0:
        print(f"    ... đang kết nối máy chủ ({attempt + 1}/45)")

if not active_port:
    print("[❌] LỖI: Server không phản hồi sau 90 giây. Chi tiết nhật ký log:")
    if os.path.exists(server_log_path):
        with open(server_log_path, "r", errors="ignore") as f:
            print(f.read()[-1500:])
    raise RuntimeError("Server failed to start in time")

print(f"[✓] Server đã READY và phản hồi thành công trên cổng: {active_port}!\n")

# 3. KIỂM TRA & CÀI ĐẶT CLOUDFLARED
if shutil.which("cloudflared"):
    print("[✓] Cloudflare Tunnel (cloudflared) đã có sẵn trên hệ thống.")
else:
    print("[*] Đang tải và cài đặt cloudflared...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
    print("[✓] Đã cài đặt cloudflared thành công.")

# 4. HÀM KIỂM TRA ĐỘ SỐNG CỦA URL
def verify_url(url, timeout=8):
    try:
        req = urllib.request.Request(
            url,
            headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        )
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            if resp.status in [200, 301, 302, 304]:
                return True, resp.status
    except urllib.error.HTTPError as e:
        if e.code in [200, 301, 302, 304]:
            return True, e.code
        return True, e.code
    except Exception as e:
        return False, str(e)
    return False, "No response"

# 5. KHỞI TẠO TUNNEL (ƯU TIÊN CLOUDFLARE, FALLBACK LOCALTUNNEL)
def extract_cf_url(log_path):
    if not os.path.exists(log_path):
        return None
    try:
        with open(log_path, "r", errors="ignore") as f:
            c = f.read()
        matches = re.findall(r"https://([a-zA-Z0-9\-]+)\.trycloudflare\.com", c)
        valid = [f"https://{m}.trycloudflare.com" for m in matches if m.lower() != "api"]
        if valid:
            return valid[-1]
    except Exception:
        pass
    return None

public_url = None
tunnel_type = "Cloudflare"
tunnel_proc = None
tunnel_pass = None
cf_log = "/content/cloudflared.log"
!rm -f {cf_log}

print(f"[*] Đang khởi tạo Cloudflare Tunnel cho cổng {active_port}...")
cf_cmd = [
    "cloudflared", "tunnel",
    "--protocol", "http2",
    "--edge-ip-version", "4",
    "--url", f"http://127.0.0.1:{active_port}"
]
tunnel_proc = subprocess.Popen(cf_cmd, stdout=open(cf_log, "w"), stderr=subprocess.STDOUT)

for attempt in range(12):
    time.sleep(1.5)
    public_url = extract_cf_url(cf_log)
    if public_url:
        break

if not public_url:
    print("[!] CẢNH BÁO: Cloudflare Tunnel chưa phản hồi. Đang chuyển sang Localtunnel...")
    try:
        tunnel_proc.terminate()
    except Exception:
        pass

    try:
        tunnel_pass = urllib.request.urlopen("https://ipv4.icanhazip.com", timeout=4).read().decode().strip()
    except Exception:
        tunnel_pass = ""

    lt_log = "/content/localtunnel.log"
    !rm -f {lt_log}
    lt_cmd = ["npx", "--yes", "localtunnel", "--port", str(active_port)]
    tunnel_proc = subprocess.Popen(lt_cmd, stdout=open(lt_log, "w"), stderr=subprocess.STDOUT)
    tunnel_type = "Localtunnel"

    for _ in range(25):
        time.sleep(1.5)
        if os.path.exists(lt_log):
            with open(lt_log, "r", errors="ignore") as f:
                content = f.read()
                m = re.search(r"https://[a-zA-Z0-9\-]+\.loca\.lt", content)
                if m:
                    public_url = m.group(0)
                    break

# 6. KIỂM TRA TRUY CẬP VÀ HIỂN THỊ LINK
colab_direct_url = None
try:
    from google.colab.output import eval_js
    colab_direct_url = eval_js(f"google.colab.kernel.proxyPort({active_port})")
except Exception:
    pass

if public_url:
    print(f"[*] Kiểm tra phản hồi thực tế của URL: {public_url}...")
    time.sleep(2)
    is_ok, check_msg = verify_url(public_url)
    if is_ok:
        print(f"[✓] Đã kết nối tới URL thành công!")
else:
    if colab_direct_url:
        public_url = colab_direct_url
        tunnel_type = "Google Colab Direct"

print("\n" + "=" * 72)
print("🚀 STUDIO VIDEO QUIZ ĐÃ KHỞI CHẠY THÀNH CÔNG VỚI BỘ TĂNG TỐC GPU!")
print("=" * 72)

if public_url:
    print(f"\n👉 PUBLIC URL TRUY CẬP STUDIO CỦA BẠN ({tunnel_type}):")
    print(f"   🔗 {public_url}\n")

if tunnel_type == "Localtunnel" and tunnel_pass:
    print(f"🔑 Mật khẩu Tunnel IP (nếu trang web yêu cầu): {tunnel_pass}\n")

if colab_direct_url and colab_direct_url != public_url:
    print(f"🔗 Link Google Colab Proxy (Dự phòng ổn định):\n   {colab_direct_url}\n")

try:
    btn_html = f'''
        <div style="background:#0f172a;border:2px solid #00e5ff;border-radius:12px;padding:22px;text-align:center;margin:15px 0;">
            <h3 style="color:#ffffff;margin:0 0 12px 0;">🎬 Studio Video Quiz Đang Chạy Trên Colab GPU</h3>
            <a href="{public_url}" target="_blank" style="display:inline-block;background:#00e5ff;color:#000000;font-weight:bold;font-size:16px;padding:12px 28px;border-radius:8px;text-decoration:none;box-shadow:0 0 15px rgba(0,229,255,0.4);margin:6px;">
                👉 BẤM VÀO ĐÂY ĐỂ MỞ TOOL TRÊN TRÌNH DUYỆT
            </a>
            <p style="color:#94a3b8;font-size:13px;margin:8px 0 0 0;">Nếu gặp màn hình cảnh báo Tunnel, bấm <b>Click to Continue</b> hoặc nhập IP <code>{tunnel_pass or ''}</code></p>
        </div>
    '''
    display(HTML(btn_html))
except Exception:
    pass

# 7. GIỮ PROCESS SỐNG LIÊN TỤC TRONG FOREGROUND
print("[*] Studio và Tunnel đang chạy ở chế độ nền liên tục. Đừng bấm nút dừng (Stop) Cell này khi đang làm việc.")
try:
    while True:
        time.sleep(30)
        if server_proc.poll() is not None:
            print("[❌] Server đã dừng đột ngột! Kiểm tra log:")
            if os.path.exists(server_log_path):
                with open(server_log_path, "r", errors="ignore") as f:
                    print(f.read()[-1000:])
            break
except KeyboardInterrupt:
    print("\n[!] Đã nhận tín hiệu dừng từ người dùng. Đang tắt server và tunnel...")
    server_proc.terminate()
    if tunnel_proc:
        tunnel_proc.terminate()
    print("[✓] Đã dọn dẹp xong. Session an toàn kết thúc.")
